# 02 – Ügyfélszegmentáció (Customer Segmentation)

**Bemenet:** `data/processed/online_retail_ready_for_rfm.parquet`  
**Kimenet:** `data/processed/rfm_features.parquet`, `models/scaler_rfm.joblib`

*Az adatok előkészítése a `01_data_preparation.ipynb` notebookban történt.*

**Tartalom:**

2\. Feature Engineering és ML keretrendszer (Time-Window)

3\. Statisztikai outlier-kezelés és skálázás

4\. K-means klaszterezés: Szegmentálás és a szegmensek profilozása.

5\. Kiterjesztett EDA: A klaszterek vizualizálása (pl. Snake plot vagy Heatmap).

In [ ]:
# ============================================================
# Konfiguráció és könyvtárak betöltése
# ============================================================
import pandas as pd
import numpy as np
from config import (
    READY_FOR_RFM_PARQUET, RFM_FEATURES_PARQUET,
    IMAGES_DIR, MODELS_DIR, SCALER_PATH, CUTOFF_DATE
)

# Mappastruktúra biztosítása
IMAGES_DIR.mkdir(parents=True, exist_ok=True)
MODELS_DIR.mkdir(parents=True, exist_ok=True)

# Előző notebook kimenetének betöltése
df = pd.read_parquet(READY_FOR_RFM_PARQUET)
print(f"Betöltve: {READY_FOR_RFM_PARQUET}")
print(f"Sorok: {len(df):,} | Oszlopok: {df.shape[1]}")

## 2. Feature Engineering és ML Keretrendszer (Time-Window)

A klasszikus prediktív Customer Lifetime Value (CLV) modellezés legkritikusabb pontja az **adatszivárgás (data leakage) megelőzése**. Annak érdekében, hogy egy valós üzleti szituációt szimuláljunk (ahol a múltbeli adatokból próbáljuk megjósolni a jövőt), az adathalmazt egy időbeli vágási pont (Cutoff Date) mentén **két szigorúan elkülönülő ablakra bontjuk**:

1. **Megfigyelési ablak (X - Observation Window):** Az adathalmaz kezdete és a vágási pont közötti időszak. Kizárólag ezekből az adatokból számítjuk ki az ügyfelek RFM (Recency, Frequency, Monetary) és egyéb viselkedési mutatóit. A gépi tanulási modell csak ezt fogja látni.
2. **Célablak (y - Target Window):** A vágási ponttól az adathalmaz végéig tartó időszak (kb. az utolsó 90 nap). Itt számoljuk ki az ügyfelek tényleges értékét (a célváltozót), amit a modellnek majd prediktálnia kell.

*Kiválasztott vágási pont:* **2011. szeptember 9.** (Mivel az adathalmaz 2011. december 9-én ér véget, ez pontosan egy 90 napos (negyedéves) előrejelzési ablakot biztosít, ami iparági sztenderd a B2B/B2C szegmentációban).

In [ ]:
# ============================================================
# 2.1 - Időablakok (Time-Windows) felállítása és szétválasztása
# ============================================================

print("Időablakok szétválasztása az adatszivárgás elkerülése érdekében...")

# Vágási pont (Cutoff date) definiálása: pontosan 90 nappal az adathalmaz vége előtt
CUTOFF_DATE_TS = pd.to_datetime(CUTOFF_DATE)

# 2.1. Megfigyelési ablak (X feature-ök alapja)
df_obs = df[df['InvoiceDate'] < CUTOFF_DATE_TS].copy()

# 2.2. Célablak (y célváltozó alapja)
df_target = df[df['InvoiceDate'] >= CUTOFF_DATE_TS].copy()

print("-" * 50)
print(f"Teljes vizsgált időszak:  {df['InvoiceDate'].min().date()}  --->  {df['InvoiceDate'].max().date()}")
print(f"Vágási pont (Cutoff):     {CUTOFF_DATE_TS.date()}")
print("-" * 50)
print(f"Megfigyelési ablak (X):   {len(df_obs):,} sor")
print(f"Célablak (y):             {len(df_target):,} sor")

# 2.3 Ellenőrizzük, hogy hány olyan ügyfél van a megfigyelési ablakban, akit érdemes vizsgálni
valos_ugyfelek_szama = df_obs['Customer ID'].nunique()
print(f"\nModellezhető egyedi ügyfelek száma a megfigyelési ablakban: {valos_ugyfelek_szama:,}")

### 2.2 Kiterjesztett RFM Feature Engineering (csak az X ablakon)

A megfigyelési ablak (`df_obs`) felhasználásával kiszámítjuk a vásárlók egyedi profilját leíró mutatókat. A hagyományos RFM modellt **kiterjesztjük a visszaküldési (return) metrikákkal**, mivel a magas sztornóarány kritikus indikátora a jövőbeli lemorzsolódásnak (churn) és a csökkenő élettartam-értéknek (CLV).

**Kiszámított Feature-ök:**
* `recency_days`: Utolsó vásárlás óta eltelt napok száma (a vágási ponttól visszaszámolva).
* `frequency`: Sikeres (pozitív) vásárlási tranzakciók (Invoice) száma.
* `monetary_total`: Nettó elköltött összeg (a visszaküldések értékével csökkentve).
* `monetary_avg`: Átlagos nettó kosárérték ($monetary\_total / frequency$).
* `return_count`: Visszaküldött (sztornó) rendelések száma.
* `return_ratio`: Visszaküldések aránya az összes aktivitáshoz képest.

*Kritikus ML Best Practice:* Minden aggregációt szigorúan a `df_obs` adathalmazon hajtunk végre, így a modell semmilyen információt nem kap a jövőbeli (vágási pont utáni) viselkedésről.

In [ ]:
# ============================================================
# 2.2 - Kiterjesztett RFM metrikák kiszámítása
# ============================================================

print("Ügyfélszintű RFM és Return feature-ök kiszámítása a megfigyelési ablakból...\n")

# 2.2.1. Pozitív tranzakciók (Vásárlások) szűrése a Recency és Frequency számításhoz
purchases = df_obs[df_obs['Quantity'] > 0]

rfm = purchases.groupby('Customer ID').agg(
    # Recency: Cutoff dátum - utolsó vásárlás dátuma
    recency_days=('InvoiceDate', lambda x: (CUTOFF_DATE_TS - x.max()).days),
    # Frequency: Egyedi számlák (Invoice-ok) száma
    frequency=('Invoice', 'nunique')
)

# 2.2.2. Monetary: Nettó költés (Vásárlások + Sztornók együttes összege az ablakban)
monetary = df_obs.groupby('Customer ID')['LineTotal'].sum().rename('monetary_total')
rfm = rfm.join(monetary)

# 2.2.3. Visszaküldések (Returns) azonosítása és számolása
returns = df_obs[df_obs['Quantity'] < 0]
return_counts = returns.groupby('Customer ID')['Invoice'].nunique().rename('return_count')

# Balra csatlakozás (Left join): akinek nincs visszaküldése, annál a NaN-t 0-ra cseréljük
rfm = rfm.join(return_counts).fillna({'return_count': 0})

# 2.2.4. Származtatott (Derived) mutatók kiszámítása
rfm['monetary_avg'] = rfm['monetary_total'] / rfm['frequency']
rfm['return_ratio'] = rfm['return_count'] / (rfm['frequency'] + rfm['return_count'])

# 2.2.5. QA: Extrém esetek szűrése (pl. aki a megfigyelési ablakban összességében mínuszban van)
# (Ez előfordulhat, ha valaki csak visszaküldött az X ablakban)
rfm = rfm[rfm['monetary_total'] > 0]

print("Feature Engineering sikeresen befejeződött.")
print(f"Létrejött RFM mátrix dimenziói: {rfm.shape[0]:,} ügyfél, {rfm.shape[1]} feature")
print("-" * 50)
display(rfm.head())

## 3. Statisztikai Outlier-kezelés és skálázás

A Feature Engineering során létrehozott RFM változók jellemzően erősen jobbra ferde (right-skewed) eloszlást mutatnak: a vásárlók nagy része kis értékben és ritkán vásárol, míg egy szűk réteg (a "Bálnák") extrém magas frekvenciát és költést produkál.

Mivel a következő lépésben K-means klaszterezést alkalmazunk – amely az Euklideszi távolságokra épül, és így rendkívül érzékeny a szélsőértékekre és a léptékkülönbségekre –, elengedhetetlen a változók eloszlásának vizsgálata, normalizálása és skálázása az algoritmus betanítása előtt.

### 3.1 Az eloszlások vizuális diagnosztikája
Első lépésként megvizsgáljuk az alapvető RFM mutatók (Recency, Frequency, Monetary) eloszlását boxplot ábrák segítségével, hogy felmérjük a statisztikai outlierek mértékét.

In [ ]:
# ============================================================
# 3.1 - RFM változók eloszlásának vizsgálata
# ============================================================
import matplotlib.pyplot as plt
import seaborn as sns

print("RFM változók eloszlásának vizualizálása...")

# Csak az alap RFM feature-öket nézzük a diagnosztikához
features_to_plot = ['recency_days', 'frequency', 'monetary_total']

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle('RFM Változók Eloszlása (Nyers adatok)', fontsize=16)

for i, col in enumerate(features_to_plot):
    sns.boxplot(x=rfm[col], ax=axes[i], color='skyblue')
    axes[i].set_title(f'{col} (Boxplot)')
    axes[i].set_xlabel('')

plt.tight_layout()

# Kép mentése dokumentációs céllal
fig_path_rfm = IMAGES_DIR / "rfm_raw_distributions.png"
plt.savefig(fig_path_rfm, bbox_inches="tight")
print(f"Ábra mentve: {fig_path_rfm}")
plt.show()

# Gyors statisztika a ferdeségről (Skewness)
# Ha az érték > 1 vagy < -1, az eloszlás erősen ferde
print("\nVáltozók ferdesége (Skewness):")
display(rfm[features_to_plot].skew().to_frame(name='Skewness').round(2))

### 3.2 Log-transzformáció és standardizálás

Az extrém magas ferdeségi (Skewness) értékek miatt a K-means klaszterezés előtt az alábbi kétlépcsős adat-előkészítést hajtjuk végre:
1. **Log-transzformáció (`np.log1p`):** Megtartjuk a legértékesebb vásárlóinkat ("Bálnák"), de a logaritmikus skálázással "összenyomjuk" a kiugró értékeket, így az eloszlás közelebb kerül a normál eloszláshoz. A `log1p` (log(x+1)) használata azért biztonságos, mert kezeli a 0 értékeket is (pl. 0 napos recency).
2. **Standardizálás (`StandardScaler`):** Mivel a K-means Euklideszi távolságot számol, a változókat azonos dimenzióba (átlag = 0, szórás = 1) kell hozni. Ennek hiányában a nagyobb számosságú metrikák (pl. Monetary) elnyomnák a kisebbeket (pl. Frequency).

*Megjegyzés: A klaszterezéshez csak a hagyományos R, F, M változókat használjuk. A visszaküldési (return) metrikák később, az XGBoost modellnél kapnak főszerepet.*

In [ ]:
# ============================================================
# 3.2. - Log-transzformáció és Skálázás
# ============================================================
from sklearn.preprocessing import StandardScaler
import joblib

print("Log-transzformáció és standardizálás folyamatban...")

# Csak az R, F, M változókat klaszterezzük
rfm_features = ['recency_days', 'frequency', 'monetary_total']
rfm_cluster_data = rfm[rfm_features].copy()

# 3.2.1. Lépés: Log-transzformáció a ferdeség csökkentésére
rfm_log = np.log1p(rfm_cluster_data)

# Nézzük meg, mennyit javult a ferdeség (Skewness)
print("\nFerdeség (Skewness) a Log-transzformáció UTÁN:")
display(rfm_log.skew().to_frame(name='Skewness').round(2))

# 3.2.2. Lépés: Standardizálás
scaler = StandardScaler()
rfm_scaled_array = scaler.fit_transform(rfm_log)

# Visszaalakítjuk DataFrame-be, hogy megmaradjon a Customer ID index
rfm_scaled = pd.DataFrame(
    rfm_scaled_array, 
    index=rfm_log.index, 
    columns=rfm_features
)

# 3.2.3. Lépés: A Scaler objektum elmentése (Kritikus lépés a Streamlit miatt!)
joblib.dump(scaler, SCALER_PATH)

print("-" * 50)
print(f"✔️ StandardScaler sikeresen illesztve és mentve ide: {SCALER_PATH}")
print("A skálázott (K-means bemeneti) adatok első 5 sora:")
display(rfm_scaled.head())

In [ ]:
# ============================================================
# 3.3. - RFM feature mátrix mentése a következő notebookhoz
# ============================================================

# Az összes feature-t (nyers RFM + return metrikák + skálázott RFM) egyetlen táblában mentjük
rfm_export = rfm.copy()
rfm_export[['recency_scaled', 'frequency_scaled', 'monetary_scaled']] = rfm_scaled.values

rfm_export.to_parquet(RFM_FEATURES_PARQUET, compression="snappy")
print(f"✔️ RFM feature mátrix mentve: {RFM_FEATURES_PARQUET}")
print(f"   Dimenziók: {rfm_export.shape[0]:,} ügyfél, {rfm_export.shape[1]} oszlop")

# 4. K-means Klaszterezés

*(Következő lépés – folytatás a portfólió következő iterációjában.)*